## Time Series Grid SANOS Market Generation Test

This notebook computes historic time series for implied vols, fitting error, DLV vs a gixed NMS grid.



In [1]:

import os
os.environ["JAX_ENABLE_X64"] = "True"

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree
%aimport quantdata.widb.options_filter
%aimport quantdata.widb.timeseries
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.sanos.sanos_tools
%aimport quantdata.sanos.arbfree
%aimport quantdata.sanos.sanos
%aimport quantdata.sanos.sanos_jax
%aimport quantdata.dysanos.grid.sanos_torch_model
%aimport quantdata.dysanos.grid.interpolated


In [ ]:

from quantdata.dysanos.grid.interpolated import GridConfig, interpolate_grid, interpolate_time_series, Context
from quantdata.biz_calendar import DateRange
from quantdata.pool import PoolConfig
from datetime import date

pool_config = PoolConfig(num_workers=8, threading=False)
grid_config = GridConfig(smoothness=0.25, mode="nms")
ts_nms = interpolate_time_series( "SPX", DateRange(start=date(2025, 1, 1), end=date(2025, 8, 30)), grid_config=grid_config, pool_config=pool_config, verbose=Context.all )
print(f"/done - found {len(ts_nms['dates'])} dates.")


00: Fitting SPX date 8/164 2025-01-08: no data after filtering, skipping.
00: Fitting SPX date 144/164 2025-08-01: done; so far this took 4s...

In [ ]:
pool_config = PoolConfig(num_workers=8, threading=False)
grid_config = GridConfig(smoothness=0.25, mode="strikes")
ts_strikes = interpolate_time_series( "SPX", DateRange(start=date(2025, 1, 1), end=date(2025, 8, 30)), grid_config=grid_config, pool_config=pool_config, verbose=Context.all )
print(f"/done - found {len(ts_strikes)} dates with data.")


In [ ]:
S, T, N    = ts['mids'].shape
compressor_config = SanosCompressorConfig(
    num_state = 10, 
    state_config = FeedForward.Config(
        activation = "ReLU",
        num_depth = 2,
        num_width = 200,
        use_bias = True,
    )
)
compressor = SanosCompressor( env,
                              S=S,
                              T=T,
                              N=N,
                              config=compressor_config )

error, _ = compressor( data, training=False )
print(error)

n_epochs = 5000

optimizer = torch.optim.AdamW(
    compressor.parameters(),
    lr=1E-3,
    weight_decay=1e-4,
)

tme = Context().timer()

for epoch in range(n_epochs):
    # -------------------------
    # Training
    # -------------------------
    compressor.train()

    train_loss = 0.0
    n_train = 0

    optimizer.zero_grad(set_to_none=True)
    loss = compressor(data, training=True)

    loss.backward()

    # Optional but often useful
    torch.nn.utils.clip_grad_norm_(compressor.parameters(), max_norm=1.0)

    optimizer.step()
    train_loss = loss.detach().item() 

    if epoch in [0,n_epochs-1] or (epoch+1) % 10 == 0:
        print(f"\rEpoch {epoch+1}/{n_epochs}, Loss: {train_loss:.4g}. So far this took {tme}.", end='', flush=True)

final_error, final_data = compressor( data, training=False )
final_calls = final_data['calls'].cpu().numpy()
xdensities = final_data['xdensities'].cpu().numpy()
dlvsqs = final_data['dlvsqs'].cpu().numpy()
print(f"\rFinal error: {final_error:.4g}. This took {tme}.")


In [ ]:
def display(ticker, benchmark_config):
    str_smooth = f"{benchmark_config.smoothness:.2f}-smooth" if benchmark_config.smoothness is not None else "Linear"

    dates   = ts['dates']
    S, T, N = ts['mids'].shape
    state   = compressor.dlv_state.detach().cpu().numpy()

    with figure(f"Compression {ticker} {str_smooth} Fits - state", col_size=4, row_size=6, columns=state.shape[1] ) as fig:
        colors = colors_tableau()
        for istate in range(state.shape[1]):
            ax = fig.add_subplot(f"state {istate}")
            ax.plot(dates, state[:,istate], color=next(colors))
            ax.set_xlabel("Date")
            ax.tick_params(axis="x", labelrotation=90)

    for ix, d in enumerate(dates):
        with figure(f"Compression {ticker} {str_smooth} Fits date #{ix} {d}", col_size=6, row_size=6, columns=None ) as fig:

            final_calls_t = final_calls[ix,:,:]
            expiries_t    = ts['expiries'][ix,:]
            xstrikes_t    = ts['xstrikes'][ix,:,:]
            mids_t        = ts['mids'][ix,:,:]
            nms_t         = ts['nms'][ix,:]
            mid_vols_t    = ts['mid_vols'][ix,:,:]
            strikes_t     = xstrikes_t[:,1:-1]
            final_vols_t  = bs.implied( k=strikes_t, prices=final_calls_t, sqrtT=np.sqrt(expiries_t[:,None]) )


            X, Y = np.meshgrid(nms_t, expiries_t*DAYS_PER_YEAR)
            ax = fig.add_subplot("Interpolated Market Vols", projection="3d")
            ax.plot_surface(X, Y, mid_vols_t, cmap='viridis', edgecolor='none')
            ax.set_ylabel("Days")
            ax.set_xlabel("NMS")

            X, Y = np.meshgrid(nms_t, expiries_t*DAYS_PER_YEAR)
            ax = fig.add_subplot("Fitted Vols", projection="3d")
            ax.plot_surface(X, Y, final_vols_t, cmap='viridis', edgecolor='none')
            ax.set_ylabel("Days")
            ax.set_xlabel("NMS")

            X, Y = np.meshgrid(nms_t, expiries_t*DAYS_PER_YEAR)
            ax = fig.add_subplot("Abs Vols Difference", projection="3d")
            ax.plot_surface(X, Y, np.abs(final_vols_t - mid_vols_t), cmap='viridis', edgecolor='none')
            ax.set_ylabel("Days")
            ax.set_xlabel("NMS")
            ax.set_zlim( 0., 0.05 )

            ax = fig.add_subplot("DLVs", projection="3d")
            x = np.reshape(nms_t[None,:] + strikes_t*0.,(-1,))
            y = np.reshape((expiries_t*DAYS_PER_YEAR)[:,None] + strikes_t*0.,(-1,))
            z = np.reshape(np.sqrt(dlvsqs[ix,:,:]),(-1,))
            dx = (x.max() - x.min())/100.
            dy = (y.max() - y.min())/100.
            p = ax.bar3d(x=x, y=y, z=0.*z, dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
            ax.set_xlabel("NMS")
            ax.set_ylabel("Log-Days")
            ax.set_zlabel("DLV")

            ax = fig.add_subplot("Density", projection="3d")
            x = np.reshape(nms_t[None,:] + strikes_t*0.,(-1,))
            y = np.reshape((expiries_t*DAYS_PER_YEAR)[:,None] + strikes_t*0.,(-1,))
            z = np.reshape(xdensities[ix,:,1:-1],(-1,))
            dx = (x.max() - x.min())/100.
            dy = (y.max() - y.min())/100.
            p = ax.bar3d(x=x, y=y, z=0.*z, dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
            ax.set_xlabel("NMS")
            ax.set_ylabel("Log-Days")
            ax.set_zlabel("Density")

            ax = fig.add_subplot("Density", projection="3d")
            x = np.reshape(nms_t[None,:] + strikes_t*0.,(-1,))
            y = np.reshape((expiries_t*DAYS_PER_YEAR)[:,None] + strikes_t*0.,(-1,))
            z = np.reshape(xdensities[ix,:,1:-1],(-1,))
            dx = (x.max() - x.min())/100.
            dy = (y.max() - y.min())/100.
            p = ax.bar3d(x=x, y=y, z=0.*z, dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
            ax.set_xlabel("NMS")
            ax.set_ylabel("Log-Days")
            ax.set_zlabel("Density")

            ax = fig.add_subplot("State")
            ax.bar( np.arange(state.shape[-1]), state[ix,:], width=0.2 )


display( "SPX", benchmark_config )
